# f696 原 alpha 标签与纯 SOXL 相对收益：目标身份审计

只读原39项 Qlib 训练价格、共同成熟时钟与精确 f696 A0 保存的 alpha 分数。逐日比较原参考组合 alpha 与纯 SOXL−SOXX H5 标签，不改模型或账户。

In [1]:
from pathlib import Path
import hashlib, json, os, shlex
import numpy as np
import pandas as pd
import qlib
from qlib.data import D
from qlib.data.dataset import DatasetH
from qlib.data.dataset.handler import DataHandlerLP
from qlib.workflow import R
from sklearn.metrics import roc_auc_score
from src.data.nas_market_data import _settings, _ssh, archive_nas_recorder
os.environ["MLFLOW_ALLOW_FILE_STORE"] = "true"
root=Path.cwd();card=root/"research/stage1_f696_alpha_target_identity_20261023"
plan=json.loads((card/"plan.json").read_text());source=plan["source"]
sha=lambda p: hashlib.sha256(Path(p).read_bytes()).hexdigest()
assert plan["frozen"] and not (card/"summary.json").exists()
for relative,expected in [
 (source["parent_card"]+"/summary.json",source["parent_summary_sha256"]),
 (source["f696_card"]+"/summary.json",source["f696_summary_sha256"]),
 ("src/workflow/cash_floor_relative_feature_reintegration.py",source["label_definition_module_sha256"]),
 (source["provider_uri"]+"/qlib_store_manifest.json",source["provider_manifest_sha256"])]:
 assert sha(root/relative)==expected,relative
parent_card=root/source["parent_card"];f696_card=root/source["f696_card"]
parent_summary=json.loads((parent_card/"summary.json").read_text())
assert parent_summary["recorder_id"]==source["parent_recorder_id"]
assert parent_summary["nas_evidence"]["sha256"]==source["parent_NAS_sha256"]
assert sha(f696_card/"qlib_recorder"/source["f696_experiment_id"]/source["f696_recorder_id"]/"artifacts/inputs/frame")==source["f696_frame_sha256"]
qlib.init(provider_uri=str((root/source["provider_uri"]).resolve()),region="us",kernels=1)
calendar=pd.DatetimeIndex(D.calendar(start_time="2011-01-03",end_time="2026-10-09",freq="day"))
with R.uri_context(str(parent_card/"qlib_recorder")):
 rec=R.get_recorder(experiment_id=source["parent_experiment_id"],recorder_id=source["parent_recorder_id"])
 frame=rec.load_object("inputs/daily_frame");timing=rec.load_object("inputs/timing")
assert frame.index.equals(calendar) and timing.index.equals(calendar)
with R.uri_context(str(f696_card/"qlib_recorder")):
 score_frame=R.get_recorder(experiment_id=source["f696_experiment_id"],recorder_id=source["f696_recorder_id"]).load_object("inputs/frame")
score=pd.Series(score_frame.alpha_probability.to_numpy(float),index=pd.DatetimeIndex(score_frame.signal_date),name="f696_alpha_probability")
assert len(score)==944 and score.index.is_unique
assert score.index.equals(frame.loc["2023-01-03":"2026-10-07"].index)
def h5(series):
 return sum(np.log1p(series.astype(float).shift(-i).clip(lower=-.99)) for i in range(5))
soxl=frame.soxl_forward_open_return;soxx=frame.soxx_forward_open_return
reference_calc=h5(.27*soxl+.63*soxx)-h5(soxx)
reference_saved=frame.relative_alpha_5d.astype(float)
valid=reference_calc.notna()&reference_saved.notna()
assert int(valid.sum())>=3960
repair=json.loads((card/"precision_repair_plan.json").read_text())
assert repair["frozen"] and repair["no_label_or_policy_selection"]
np.testing.assert_allclose(reference_calc[valid].to_numpy(),reference_saved[valid].to_numpy(),rtol=0,atol=3e-8)
pure=h5(soxl)-h5(soxx)
assert pure.notna().sum()==reference_saved.notna().sum()
maturity=pd.to_datetime(timing.return_available_at.shift(-4),utc=True)
fixed=plan["fixed_definitions"]
train=((frame.index>=pd.Timestamp(fixed["train_signals"][0]))&(frame.index<=pd.Timestamp(fixed["train_signals"][1]))&
       (maturity<pd.Timestamp(fixed["train_label_available_before"]).tz_convert("UTC"))&
       reference_saved.notna()&pure.notna())
held=((frame.index>=pd.Timestamp(fixed["reused_held_signals"][0]))&(frame.index<=pd.Timestamp(fixed["reused_held_signals"][1]))&
      (maturity<pd.Timestamp(fixed["held_label_available_before"]).tz_convert("UTC"))&
      reference_saved.notna()&pure.notna())
assert int(train.sum())==fixed["expected_train_mature"] and int(held.sum())==fixed["expected_held_mature"]
static=pd.concat({"feature":pd.DataFrame({"f696_alpha_probability":score.reindex(frame.index)}),
                  "label":pd.DataFrame({"reference_alpha":reference_saved,"pure_relative":pure}),
                  "meta":pd.DataFrame({"train_mature":train,"held_mature":held})},axis=1)
static.index=pd.MultiIndex.from_arrays([frame.index,["SOXL_SOXX_CASH"]*len(frame)],names=["datetime","instrument"])
dataset=DatasetH(handler=DataHandlerLP.from_df(static),segments={"train":fixed["train_signals"],"held":fixed["reused_held_signals"]})
tr=dataset.prepare("train",col_set=["feature","label","meta"])
he=dataset.prepare("held",col_set=["feature","label","meta"])
tr=tr.loc[tr[("meta","train_mature")].astype(bool)]
he=he.loc[he[("meta","held_mature")].astype(bool)]
assert len(tr)==3014 and len(he)==940
labels=pd.DataFrame({"signal_date":frame.index,"reference":reference_saved,"pure":pure,
                     "f696_alpha_score":score.reindex(frame.index),"train_mature":train,"held_mature":held})
def table(mask):
 x=labels.loc[mask].copy();a=x.reference.gt(0);b=x.pure.gt(0)
 return {"rows":len(x),"reference_positive":int(a.sum()),"pure_positive":int(b.sum()),
         "both_positive":int((a&b).sum()),"both_nonpositive":int((~a&~b).sum()),
         "reference_only_positive":int((a&~b).sum()),"pure_only_positive":int((~a&b).sum()),
         "disagree":int((a!=b).sum()),"disagree_fraction":float((a!=b).mean())}
train_result=table(train);held_result=table(held)
def auc(value,target):
 y=target.gt(0).astype(int)
 return float(roc_auc_score(y,value)) if y.nunique()==2 else None
held_labels=labels.loc[held].copy()
full_auc={"reference":auc(held_labels.f696_alpha_score,held_labels.reference),
          "pure":auc(held_labels.f696_alpha_score,held_labels.pure)}
years={str(year):{**table(held & (labels.signal_date.dt.year.to_numpy()==year)),
                  "AUC_reference":auc(x.f696_alpha_score,x.reference),"AUC_pure":auc(x.f696_alpha_score,x.pure),
                  "partial_endpoint":year in (2023,2026)}
       for year,x in held_labels.groupby(held_labels.signal_date.dt.year)}
quarters={str(q):{**table(held & (labels.signal_date.dt.to_period("Q").astype(str).to_numpy()==str(q))),
                  "AUC_reference":auc(x.f696_alpha_score,x.reference),"AUC_pure":auc(x.f696_alpha_score,x.pure)}
          for q,x in held_labels.groupby(held_labels.signal_date.dt.to_period("Q"))}
assert len(years)==4 and len(quarters)==16
result={"experiment":card.name,"status":"read_only_exact_f696_alpha_target_identity",
        "plan_sha256":sha(card/"plan.json"),"source_parent_recorder_id":source["parent_recorder_id"],
        "source_f696_recorder_id":source["f696_recorder_id"],"calendar_rows":len(calendar),
        "reference_label_formula_max_abs_error":float((reference_calc[valid]-reference_saved[valid]).abs().max()),
        "train":train_result,"held":held_result,"held_AUC":full_auc,"annual":years,"quarterly":quarters,
        "new_fits":0,"new_accounts":0,"new_bootstraps":0,"future_outcome_used_as_policy":False,
        "independent_forward_evidence":False}
with R.start(experiment_name=card.name,uri=str((card/"qlib_recorder").resolve())):
 recorder=R.get_recorder()
 recorder.save_objects(artifact_path="inputs",plan=plan,dataset=dataset)
 recorder.save_objects(artifact_path="analysis",labels=labels,result=result)
 R.log_params(parent="exact_f696_A0",diagnostic_only=True)
 R.log_metrics(train_disagree_fraction=train_result["disagree_fraction"],held_disagree_fraction=held_result["disagree_fraction"])
 rid,eid=recorder.id,str(recorder.experiment_id)
result.update(recorder_id=rid,experiment_id=eid)
(card/"result.json").write_text(json.dumps(result,ensure_ascii=False,indent=2)+"\n")
binding=json.loads((parent_card/"nas_binding.json").read_text());binding.pop("evidence",None)
(card/"nas_binding.json").write_text(json.dumps(binding,ensure_ascii=False,indent=2)+"\n")
evidence=archive_nas_recorder(card/"nas_binding.json",workspace_root=root,card=card,recorder_id=rid,experiment_id=eid)
remote=_ssh(_settings(root)["ssh"],"sha256sum "+shlex.quote(evidence["nas_path"])).split()[0]
assert remote==evidence["sha256"]
summary={"experiment":card.name,"decision":"descriptive_target_mismatch_no_fit_or_promotion",
         "objective_achieved":False,"plan_sha256":result["plan_sha256"],"result":"result.json",
         "recorder_id":rid,"experiment_id":eid,"NAS_evidence":evidence,"NAS_readback":{"verified":True,"sha256":remote},
         "new_model_fits":0,"new_native_accounts":0,
         "artifact_policy":{"policy":"slim_card_recorder_first","workflow_config":"workflow.yaml",
                            "recorder_only_artifacts":["inputs/*","analysis/labels"]},
         "limitations":["Reference alpha label is a mixed portfolio versus SOXX target, not pure SOXL selection.",
                        "Label mismatch and score AUC are descriptive reused-history diagnostics, not retraining evidence.",
                        "The already closed original39 pure-relative Ridge task and SOXX economic gates remain unchanged."]}
(card/"summary.json").write_text(json.dumps(summary,ensure_ascii=False,indent=2)+"\n")
(card/"workflow.yaml").write_text("experiment: "+card.name+"\nentrypoint: "+card.name+".ipynb\nphase: read_only_target_identity\n")
(card/"conclusion.md").write_text(
 "# f696 原 alpha 标签与纯 SOXL 相对收益：目标身份审计\n\n"
 "原 Stage1 的 `relative_alpha_5d` 是27% SOXL/63% SOXX/10%现金日重置参考相对 SOXX，不是纯 SOXL−SOXX。"
 "本卡从原 Qlib 价格与时钟重建两个H5标签，并按3014个训练成熟日、940个复用评价成熟日及精确 f696 alpha 概率逐日核对；"
 "0拟合、0账户、0新行情。\n\n"
 f"训练标签符号不一致 {train_result['disagree']}/{train_result['rows']}（{train_result['disagree_fraction']:.2%}），"
 f"复用评价 {held_result['disagree']}/{held_result['rows']}（{held_result['disagree_fraction']:.2%}）。"
 f"原 f696 alpha 分数在评价期对原参考/纯相对标签的 AUC 为 {full_auc['reference']:.4f}/{full_auc['pure']:.4f}。"
 "完整年度、季度及两个部分年度均在 `result.json`，不能据最优区间改交易规则。\n\n"
 "标签差异只是训练目标身份事实，不证明纯相对标签可预测或替换主配置目标能改善完整资金账户。"
 "已关闭的原39项纯相对Ridge信息失败仍有效；若考虑改神经alpha监督，须有新的预测增量理由与独立冻结经济合同。"
 f"原生Recorder `{rid}` 和NAS SHA `{remote}` 已远端核对；SOXX稳定超额仍未获得。\n")
print(json.dumps({"train_disagree":train_result["disagree_fraction"],"held_disagree":held_result["disagree_fraction"],"held_AUC":full_auc,"recorder":rid},ensure_ascii=False))

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


[77761:MainThread](2026-10-23 23:15:10,846) INFO - qlib.Initialization - [config.py:453] - default_conf: client.


[77761:MainThread](2026-10-23 23:15:10,847) INFO - qlib.Initialization - [__init__.py:82] - qlib successfully initialized based on client settings.


[77761:MainThread](2026-10-23 23:15:10,848) INFO - qlib.Initialization - [__init__.py:84] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/minute_information_pit_allocator_20261012/daily_prices')}


/Users/rziiiii/Library/Python/3.14/lib/python/site-packages/mlflow/tracking/_tracking_service/utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)


...

/Users/rziiiii/Library/Python/3.14/lib/python/site-packages/mlflow/tracking/_tracking_service/utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)
[77761:MainThread](2026-10-23 23:15:10,976) WARNING - qlib.workflow - [expm.py:230] - No valid experiment found. Create a new experiment with name stage1_f696_alpha_target_identity_20261023.


[77761:MainThread](2026-10-23 23:15:10,977) INFO - qlib.workflow - [exp.py:258] - Experiment 149528688250622915 starts running ...


[77761:MainThread](2026-10-23 23:15:11,032) INFO - qlib.workflow - [recorder.py:345] - Recorder c67a0cb208054221b2983b01eda90388 starts running under Experiment 149528688250622915 ...


[77761:MainThread](2026-10-23 23:15:11,635) INFO - qlib.timer - [log.py:127] - Time cost: 0.001s | waiting `async_log` Done


{"train_disagree": 0.011944260119442602, "held_disagree": 0.009574468085106383, "held_AUC": {"reference": 0.506928945386325, "pure": 0.5092076713739335}, "recorder": "c67a0cb208054221b2983b01eda90388"}


## 中文结论

原参考组合alpha与纯 SOXL−SOXX 的二元标签在训练3014日仅36日、复用评价940日仅9日不同；原 f696 alpha分数对两种标签的总体 AUC 分别约0.507/0.509。2024完整年均低于0.45、2025均高于0.60，不能挑年份。决定 `closed_prefit_binary_alpha_relabel_almost_identical`：换二元目标基本没有改任务，不为此重训或运行账户。全年、季度、原生Recorder及NAS恢复证据见 `result.json`、`nas_scoped_restore.json`。